## CellCycleNet Example - Fine tune pretrained model and predict cell cycle stage on 3D DAPI images WITH ground truth labels.
This notebook demonstrates how to use CellCycleNet to fine tune the pretrained model and predict cell cycle stage from images of DAPI-stained nuclei that have associated ground truth labels for cell cycle stage.

CellCycleNet requires the following data:
 - A directory of 3D DAPI-stained fields of view named as `tile_<tile_num>.tiff`
 - A directory of 3D segmentation masks named as `mask_<tile_num>.tiff`
 - A directory of 3D ground truth label arrays named as `label_<tile_num>.npy` (pixel values must be 0, 1, or 2 where 0 = background, 1 = G1 nucleus, 2 = S/G2 nucleus)
 - Where `<tile_num>` is an integer that uniquely identifies each field of view and its corresponding segmentation mask and label array

### Step 1: Create single-nucleus images from segmented FOVs.

In [ ]:
from cellcyclenet import utils

IMAGE_DIR = '../data/test_tiles/' # path to DAPI-stained FOVs
MASK_DIR = '../data/test_masks/' # path to segmentation masks of FOVs 
LABEL_DIR = '../data/test_labels/' # path to label arrays of FOVs
OUTPUT_DIR = '../data/test_SNI_label/' # path where labeled single-nucleus images will be saved

# generate labeled SNIs #
'''
Optional Arguments for utils.generate_images_labeled():
 - return_df: boolean, if True, returns a pandas dataframe with the tile numbers, object numbers, and labels of the labeled SNIs
 - num_cores: integer, number of cores to use for parallel processing; if 'None', no parallel processing will be used
 - is_3d: boolean, set to True if your data is 3D; set to False if your data is 2D
'''
df = utils.generate_images_labeled(IMAGE_DIR, MASK_DIR, LABEL_DIR, OUTPUT_DIR, return_df=True, num_cores=None, is_3d=True)
df

### Step 2: Load pretrained model and finetune with labeled data.

In [ ]:
from cellcyclenet import CellCycleNet

# create 3D model instance (pretrained weights are loaded by default) #
model = CellCycleNet(is_3d=True)

# convert dataframe to CellCycleNet dataset; split_data=True --> 70% training, 20% validation, 10% test #
train, val, test = model.create_dataset(dataframe=df, split_data=True)

# train model on your labeled data #
'''
Optional Arguments for CellCycleNet.train():
 - transform: callable, a function that takes an image and returns a transformed image; if None, no transformation will be applied
 - lazy_load: boolean, if True, the dataset will be loaded lazily in each epoch (slower, but uses less memory); if False, the dataset will be loaded into memory (faster, but uses more memory) 
 - verbose: boolean, if True, training progress will be printed to the console
'''
model.train(train, val, n_epochs=10, batch_size=4, initial_LR=1e-5, transform=None, lazy_load=True, verbose=True)

# save finetuned model weights #
model.save_model('fine_tuned_model.pt')

### Step 3: Predict cell cycle stage for each single-nucleus image in the test set.

In [ ]:
# generate cell cycle stage predictions (0 = G1, 1 = S/G2); with_labels=True --> predicting on labeled data #
test_predictions = model.predict(test, with_labels=True)

# display predictions #
test_predictions.sort_values(['tile_num', 'obj_num'])
test_predictions

# plot an ROC curve to evaluate model peformance #
model.plot_ROC(test_predictions['label'], test_predictions['pred'], test_predictions['prob'])